<a href="https://colab.research.google.com/github/malvathrahul/AI-ML_project/blob/main/Block1_Data_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Block 1 - Synthetic Data Generation
**Project 48: AI-Based Classification of Defect Type (Pore / Crack / Inclusion) from CT Sub-Volumes**

Before running: **Runtime -> Change runtime type -> T4 GPU** (the GPU is not needed for this block, but you will need it from Block 2 on).
Then run the cells **in order, top to bottom**. Total time: about 10 minutes.

| Cell | What it does | Brief requirement it covers |
|---|---|---|
| 1 | Setup, Google Drive, all settings | formats, sizes |
| 2 | Pore / crack / inclusion generators | points 8, 9 |
| 3 | CT noise + builds one simulated scan | point 8 (Gaussian + Poisson, 3 SNR tiers) |
| 4 | Preview - look at the defects first | sanity check |
| 5 | Generate all 7,500 patches from 200 scans | point 6 |
| 6 | Save `.npy` + `.csv` to Drive | point 7 |
| 7 | Validation checks + report figures | points 9, 21 |

## Cell 1 - Setup and configuration
Mounts Google Drive (click **Connect** / allow access when asked) and defines every number from the brief in one place.

In [ ]:
# ============ CELL 1: SETUP, DRIVE, CONFIG ============
import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.stats import ks_2samp
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# --- Save location: Google Drive (so nothing is lost if Colab disconnects) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/defect_project/data'
except ImportError:                      # running somewhere other than Colab
    OUT_DIR = './defect_project/data'
os.makedirs(OUT_DIR + '/figures', exist_ok=True)

# --- Project configuration (every number here comes from the project brief) ---
SEED          = 42
N_PARENTS     = 200            # ~200 parent volumes (simulated CT scans)
N_PER_CLASS   = 2500           # 2,500 per class -> 7,500 patches in total
CLASS_NAMES   = ['pore', 'crack', 'inclusion']

GRID, CELL, MARGIN = 4, 40, 8              # each parent = 4x4x4 cells of 40^3 voxels (+8 voxel margin)
PARENT = 2 * MARGIN + GRID * CELL          # 176^3 voxels per parent volume
STORE  = 40                                # stored patch size 40^3 (model will random-crop 32^3 from it)
JITTER = 2.0                               # defect centre wobble inside its cell (voxels)

V_RANGE        = (60, 600)                 # defect volume in voxels, SAME log-uniform range for all 3 classes
CONTRAST_RANGE = (0.05, 0.85)              # |inclusion contrast| relative to matrix: subtle -> obvious
BIN_EDGES      = (0.20, 0.50)              # subtle < 0.20 <= moderate < 0.50 <= obvious
SNR_TIERS      = {'high': 40, 'medium': 20, 'low': 10}   # SNR = matrix level / noise std
PSF_RANGE      = (0.6, 0.9)                # scanner blur (Gaussian sigma, voxels) -> partial-volume effect
MATRIX_RANGE   = (0.8, 1.2)                # base attenuation of the solid material, differs per scan
SUBTYPES = {0: ['spherical', 'ellipsoidal', 'irregular'],
            1: ['planar', 'curved'],
            2: ['spherical', 'polyhedral']}

print('Output folder :', OUT_DIR)
print('Parent volume :', PARENT, '^3 voxels | stored patch:', STORE, '^3 voxels')

## Cell 2 - The three defect generators
* **Pore**: near-zero-attenuation void; spherical, ellipsoidal or irregular.
* **Crack**: thin near-zero-attenuation sheet; planar or curved (bent).
* **Inclusion**: solid sphere or polyhedron whose attenuation is **above or below** the matrix (contrast from subtle to obvious).

All three classes are drawn from the **same volume range**, so "how much void is in the patch" cannot be used as a shortcut (brief, point 21).

In [ ]:
# ============ CELL 2: THE THREE DEFECT GENERATORS ============
# Local coordinate grid of one cell (40x40x40). Defects are built around a centre point.
_g = np.arange(CELL, dtype=np.float32)
COORDS = np.stack(np.meshgrid(_g, _g, _g, indexing='ij'), axis=-1)   # (40,40,40,3)
STRUCT26 = ndi.generate_binary_structure(3, 3)

def random_rotation(rng):
    """Uniformly random 3D rotation matrix."""
    Q, R = np.linalg.qr(rng.standard_normal((3, 3)))
    Q = Q @ np.diag(np.sign(np.diag(R)))
    if np.linalg.det(Q) < 0:
        Q[:, 0] *= -1
    return Q.astype(np.float32)

def smooth_field(rng, sigma):
    """Smooth random 3D field with unit std (used to make shapes irregular)."""
    f = ndi.gaussian_filter(rng.standard_normal((CELL,) * 3).astype(np.float32), sigma)
    return f / (f.std() + 1e-8)

def smallest_V(score, V):
    """Boolean mask of EXACTLY the V voxels with the smallest score.
    This is how we hit a target volume for every shape type."""
    flat = score.ravel()
    mask = np.zeros(flat.shape, bool)
    mask[np.argpartition(flat, V - 1)[:V]] = True
    return mask.reshape(score.shape)

def largest_component(mask):
    lab, n = ndi.label(mask, structure=STRUCT26)
    if n <= 1:
        return mask
    sizes = ndi.sum(mask, lab, index=np.arange(1, n + 1))
    return lab == (1 + int(np.argmax(sizes)))

# ---------------- 1) PORES: spherical / ellipsoidal / irregular voids ----------------
def gen_pore(rng, V, subtype, center):
    r = (3 * V / (4 * np.pi)) ** (1 / 3)                    # radius of a sphere with volume V
    if subtype == 'spherical':
        ax = np.ones(3)
    elif subtype == 'ellipsoidal':
        ax = rng.uniform(1.0, 2.2, 3)                        # random elongation of each axis
    else:                                                    # irregular
        ax = rng.uniform(1.0, 1.6, 3)
    ax = ax / np.cbrt(np.prod(ax)) * r                       # keep the volume equal to V
    q = (COORDS - center) @ random_rotation(rng)
    d = np.sqrt(((q / ax) ** 2).sum(-1))                     # normalised ellipsoid distance
    if subtype == 'irregular':
        d = d + rng.uniform(0.25, 0.5) * smooth_field(rng, max(1.0, 0.5 * r))   # wobbly surface
    return smallest_V(d, V)

# ---------------- 2) CRACKS: thin planar / curved sheets ----------------
def gen_crack(rng, V, subtype, center):
    t = rng.uniform(1.2, 2.4)                                # sheet thickness (voxels)
    ab = V / t / np.pi                                       # a*b of the in-plane ellipse
    ar = rng.uniform(1.0, 1.8)
    a, b = min(np.sqrt(ab * ar), 11.0), min(np.sqrt(ab / ar), 11.0)   # keep the crack inside the patch
    t = V / (np.pi * a * b)                                  # re-compute thickness so the volume stays V
    q = (COORDS - center) @ random_rotation(rng)
    u, v, w = q[..., 0], q[..., 1], q[..., 2]
    rho = np.sqrt((u / a) ** 2 + (v / b) ** 2) * (1 + 0.25 * smooth_field(rng, 2.0))   # ragged outline
    if subtype == 'curved':                                   # bend the sheet like a curved surface
        A = rng.uniform(-3, 3, 3)
        h = A[0] * (u / a) ** 2 + A[1] * (v / b) ** 2 + A[2] * (u / a) * (v / b)
    else:
        h = 0.0
    score = np.maximum(np.abs(w - h) / (t / 2), rho)
    return smallest_V(score, V)

# ---------------- 3) INCLUSIONS: solid spheres / polyhedra ----------------
def fibonacci_sphere(n):
    i = np.arange(n) + 0.5
    phi, theta = np.arccos(1 - 2 * i / n), np.pi * (1 + 5 ** 0.5) * i
    return np.stack([np.cos(theta) * np.sin(phi), np.sin(theta) * np.sin(phi), np.cos(phi)], 1)

def gen_inclusion(rng, V, subtype, center):
    q = (COORDS - center) @ random_rotation(rng)
    if subtype == 'spherical':
        d = np.linalg.norm(q, axis=-1)
    else:                                                     # convex polyhedron with 6-14 random faces
        nf = int(rng.integers(6, 15))
        N = fibonacci_sphere(nf) + rng.normal(0, 0.15, (nf, 3))
        N = (N / np.linalg.norm(N, axis=1, keepdims=True)).astype(np.float32)
        h = rng.uniform(0.8, 1.2, nf).astype(np.float32)
        q = q / rng.uniform(1.0, 1.5, 3).astype(np.float32)   # slight elongation
        d = np.max((q @ N.T) / h, axis=-1)
    return smallest_V(d, V)

def make_defect(rng, cls, V, subtype, center):
    """Generate a defect mask; retry if it is broken or too close to the cell edge."""
    gens = {0: gen_pore, 1: gen_crack, 2: gen_inclusion}
    for _ in range(10):
        m = largest_component(gens[cls](rng, V, subtype, center))
        inside = m[4:-4, 4:-4, 4:-4].sum() == m.sum()         # stays away from cell borders
        if m.sum() >= 0.85 * V and inside:
            return m
    return m
print('Generators ready: pore / crack / inclusion')

## Cell 3 - CT noise and one simulated scan ("parent volume")
Each parent volume is a 176^3 scan with its own matrix attenuation, scanner blur (partial-volume effect), mild shading, and one of three noise levels (**SNR 40 / 20 / 10**), mixing Poisson and Gaussian noise. Defects sit in separate cells, so they never overlap. Noise is added to the whole scan **before** patches are cut out.

In [ ]:
# ============ CELL 3: CT NOISE + BUILD ONE PARENT VOLUME (= one simulated scan) ============
def add_ct_noise(vol, matrix, snr, rng, corr_sigma=0.6):
    """Poisson (photon) + Gaussian (electronic) noise, slightly blurred like reconstructed CT noise.
    SNR = matrix level / noise std (measured in the plain matrix)."""
    sigma = matrix / snr
    k = matrix / (0.5 * sigma ** 2)                      # photon counts per unit attenuation -> half the variance
    pois = rng.poisson(np.clip(vol, 0, None) * k).astype(np.float32) / k
    gauss = rng.standard_normal(vol.shape, dtype=np.float32) * np.float32(sigma * np.sqrt(0.5))
    noise = (pois - vol) + gauss
    smooth = ndi.gaussian_filter(noise, corr_sigma)      # CT noise is spatially correlated
    smooth *= noise.std() / (smooth.std() + 1e-12)       # keep the intended noise level
    return vol + smooth

def contrast_bin(cls, c):
    if cls < 2:
        return 'void'
    a = abs(c)
    return 'subtle' if a < BIN_EDGES[0] else ('moderate' if a < BIN_EDGES[1] else 'obvious')

SHELL = np.ones((STORE,) * 3, bool); SHELL[4:-4, 4:-4, 4:-4] = False      # outer 4-voxel shell = local background

def build_parent(spec):
    rng = np.random.default_rng(spec['seed'])
    m = spec['matrix']
    vol = np.full((PARENT,) * 3, m, np.float32)
    lab = np.zeros((PARENT,) * 3, np.int16)                 # which defect each voxel belongs to
    cells = [(i, j, k) for i in range(GRID) for j in range(GRID) for k in range(GRID)]
    rng.shuffle(cells)
    n = spec['n_per_class']
    classes = np.array([0] * n + [1] * n + [2] * n); rng.shuffle(classes)

    defects = []
    for did, (cls, cell) in enumerate(zip(classes, cells), start=1):
        V = int(np.exp(rng.uniform(np.log(V_RANGE[0]), np.log(V_RANGE[1]))))
        sub = str(rng.choice(SUBTYPES[cls]))
        center = (CELL - 1) / 2 + rng.uniform(-JITTER, JITTER, 3)
        mask = make_defect(rng, cls, V, sub, center)
        if cls < 2:                                          # void: (almost) zero attenuation
            val = m * rng.uniform(0.0, 0.05); c = val / m - 1
        else:                                                # inclusion: brighter OR darker than matrix
            c = rng.choice([-1, 1]) * rng.uniform(*CONTRAST_RANGE); val = m * (1 + c)
        o = MARGIN + np.array(cell) * CELL
        sl = tuple(slice(int(o[a]), int(o[a]) + CELL) for a in range(3))
        vol[sl][mask] = val
        lab[sl][mask] = did
        centroid = o + np.array(ndi.center_of_mass(mask))
        defects.append(dict(did=did, cls=int(cls), sub=sub, V=V, c=float(c), centroid=centroid, vox=int(mask.sum())))

    # --- scanner effects: blur (partial volume) -> mild shading -> noise ---
    vol = ndi.gaussian_filter(vol, spec['psf'])
    g = rng.standard_normal(3); g /= np.linalg.norm(g)
    ax = [np.linspace(-0.5, 0.5, PARENT, dtype=np.float32) for _ in range(3)]
    ramp = g[0] * ax[0][:, None, None] + g[1] * ax[1][None, :, None] + g[2] * ax[2][None, None, :]
    vol = vol * (1 + spec['shade'] * ramp)
    noisy = add_ct_noise(vol, m, spec['snr'], rng)

    # --- cut one patch centred on every defect ---
    X = np.zeros((len(defects), STORE, STORE, STORE), np.float16)
    M = np.zeros((len(defects), STORE, STORE, STORE), np.uint8)
    rows = []
    for i, d in enumerate(defects):
        s = np.clip(np.round(d['centroid'] - (STORE - 1) / 2).astype(int), 0, PARENT - STORE)
        w = tuple(slice(int(s[a]), int(s[a]) + STORE) for a in range(3))
        p, lw = noisy[w], lab[w]
        X[i] = p
        M[i] = (lw == d['did'])
        rows.append(dict(
            label=d['cls'], class_name=CLASS_NAMES[d['cls']], subtype=d['sub'],
            parent_id=spec['parent_id'], split=spec['split'],
            snr_tier=spec['snr_tier'], snr_value=spec['snr'], matrix_level=round(m, 4),
            psf_sigma=round(spec['psf'], 3),
            contrast=round(d['c'], 4), abs_contrast=round(abs(d['c']), 4),
            contrast_bin=contrast_bin(d['cls'], d['c']),
            target_volume=d['V'], actual_volume=d['vox'], fill_fraction=d['vox'] / STORE ** 3,
            bg_level=round(float(np.median(p[SHELL])), 4),
            foreign_voxels=int(((lw != 0) & (lw != d['did'])).sum())))
    return X, M, rows

def make_specs(rng):
    """200 parent volumes. The split is decided HERE, per parent -> no leakage possible.
    100 parents hold 13 defects/class, 100 hold 12/class -> exactly 2,500 per class,
    and a 70/15/15 split of exactly 5,250 / 1,125 / 1,125 patches."""
    plan = [('train', 70, 70), ('val', 15, 15), ('test', 15, 15)]      # (split, #parents with 13/class, #parents with 12/class)
    specs = []
    for split, n13, n12 in plan:
        group = [dict(split=split, n_per_class=13)] * n13 + [dict(split=split, n_per_class=12)] * n12
        group = [dict(g) for g in group]
        rng.shuffle(group)
        tiers = list(SNR_TIERS)
        for i, g in enumerate(group):                                  # SNR tiers spread evenly inside every split
            g['snr_tier'] = tiers[i % 3]; g['snr'] = SNR_TIERS[tiers[i % 3]]
        specs += group
    rng.shuffle(specs)
    for pid, s in enumerate(specs):
        s['parent_id'] = pid
        s['seed'] = SEED * 1000 + pid
        s['matrix'] = float(rng.uniform(*MATRIX_RANGE))
        s['psf'] = float(rng.uniform(*PSF_RANGE))
        s['shade'] = float(rng.uniform(0.0, 0.03))
    return specs
print('Parent builder ready')

## Cell 4 - Quick preview
Builds one scan in ~2 seconds and shows the defects. **Look at the pictures before generating everything.**

In [ ]:
# ============ CELL 4: QUICK PREVIEW (one scan, takes ~2 s) -> LOOK AT THE DEFECTS BEFORE GENERATING EVERYTHING ============
def view(patch, bg):
    return patch.astype(np.float32) / bg          # divide by background so matrix = 1.0, void = 0

def plot_gallery(X, df, per_row=5, fname=None, seed=0):
    """4 rows: pore / crack / dark inclusion / bright inclusion. Central slice of each patch (matrix = 1.0)."""
    rng = np.random.default_rng(seed)
    groups = [('Pore', df.label == 0), ('Crack', df.label == 1),
              ('Inclusion (darker than matrix)', (df.label == 2) & (df.contrast < 0)),
              ('Inclusion (brighter than matrix)', (df.label == 2) & (df.contrast > 0))]
    fig, axes = plt.subplots(4, per_row, figsize=(2.6 * per_row, 11))
    for r, (title, cond) in enumerate(groups):
        idx = rng.choice(np.where(cond)[0], per_row, replace=False)
        for c, i in enumerate(idx):
            ax = axes[r, c]
            ax.imshow(view(X[i], df.bg_level.iloc[i])[STORE // 2], cmap='gray', vmin=0, vmax=1.8)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f"{df.subtype.iloc[i]}\ncontrast {df.contrast.iloc[i]:+.2f}  SNR {df.snr_value.iloc[i]}", fontsize=8)
        axes[r, 0].set_ylabel(title, fontsize=9)
    plt.suptitle('Example patches: central axial slice (matrix = 1.0, void = 0)', y=0.995)
    plt.tight_layout()
    if fname: plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

def plot_3planes(X, M, df, fname=None, seed=1):
    """One example per class, three orthogonal slices + the exact ground-truth mask."""
    rng = np.random.default_rng(seed)
    picks = [rng.choice(np.where(df.label == k)[0]) for k in range(3)]
    fig, axes = plt.subplots(3, 6, figsize=(14, 7.5))
    for r, i in enumerate(picks):
        p = view(X[i], df.bg_level.iloc[i]); m = M[i]; h = STORE // 2
        for c, (img, mk, name) in enumerate([(p[h], m[h], 'axial'), (p[:, h], m[:, h], 'coronal'), (p[:, :, h], m[:, :, h], 'sagittal')]):
            axes[r, 2 * c].imshow(img, cmap='gray', vmin=0, vmax=1.8); axes[r, 2 * c].set_title(f'{name} (CT)', fontsize=8)
            axes[r, 2 * c + 1].imshow(mk, cmap='gray'); axes[r, 2 * c + 1].set_title(f'{name} (true mask)', fontsize=8)
        axes[r, 0].set_ylabel(f"{CLASS_NAMES[df.label.iloc[i]]}\n{df.subtype.iloc[i]}\ncontrast {df.contrast.iloc[i]:+.2f}", fontsize=9)
        for a in axes[r]: a.set_xticks([]); a.set_yticks([])
    plt.tight_layout()
    if fname: plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()

# ---- build ONE scan with 13 defects per class and look at it ----
_specs = make_specs(np.random.default_rng(SEED))
t0 = time.time()
_X, _M, _rows = build_parent(_specs[1])
_df = pd.DataFrame(_rows)
print(f'One parent volume built in {time.time()-t0:.1f} s  ->  {len(_df)} patches of shape {_X.shape[1:]}')
print(_df.groupby('class_name')[['actual_volume', 'abs_contrast']].mean().round(2))
plot_gallery(_X, _df, per_row=5)
plot_3planes(_X, _M, _df)

## Cell 5 - Generate the full dataset
200 parent volumes -> 7,500 patches (2,500 per class), each stored as a 40^3 patch centred on the defect (the model will randomly crop 32^3 from it).
The train/val/test split is decided **per parent volume**, so no scan appears in two splits.

In [ ]:
# ============ CELL 5: GENERATE THE FULL DATASET (200 parent volumes -> 7,500 patches) ============
# Takes roughly 5-10 minutes. Patches are written straight into their split (train/val/test) arrays,
# split is decided per PARENT VOLUME -> volume-disjoint, no leakage possible.
SPLIT_N = {'train': 5250, 'val': 1125, 'test': 1125}          # exact 70 / 15 / 15 % of 7,500
rng_main = np.random.default_rng(SEED)
specs = make_specs(rng_main)
assert len(specs) == N_PARENTS

Xs = {s: np.zeros((n, STORE, STORE, STORE), np.float16) for s, n in SPLIT_N.items()}   # noisy CT patches
Ms = {s: np.zeros((n, STORE, STORE, STORE), np.uint8) for s, n in SPLIT_N.items()}     # exact defect masks (for the shape-only ablation)
slot = {s: rng_main.permutation(n) for s, n in SPLIT_N.items()}                          # shuffled row positions
used = {s: 0 for s in SPLIT_N}
rows_by_split = {s: [None] * n for s, n in SPLIT_N.items()}

t0 = time.time()
for spec in tqdm(specs, desc='parent volumes'):
    X, M, rows = build_parent(spec)
    s, n = spec['split'], len(rows)
    pos = slot[s][used[s]: used[s] + n]; used[s] += n
    Xs[s][pos] = X; Ms[s][pos] = M
    for p, r in zip(pos, rows):
        rows_by_split[s][p] = r
print(f'Done in {(time.time()-t0)/60:.1f} min')
assert all(used[s] == SPLIT_N[s] for s in SPLIT_N), used

dfs = []
for s in SPLIT_N:
    d = pd.DataFrame(rows_by_split[s]); d.insert(0, 'row', np.arange(len(d)))
    d.insert(0, 'patch_id', [f'{s}_{i:05d}' for i in range(len(d))])
    dfs.append(d)
df_all = pd.concat(dfs, ignore_index=True)
print(df_all.shape)
df_all.head()

## Cell 6 - Save to Google Drive
Saves patches (`.npy`), exact defect masks (`.npy`, used for the shape-only ablation), index files (`.csv`) and the settings (`.json`).

In [ ]:
# ============ CELL 6: SAVE EVERYTHING TO GOOGLE DRIVE (.npy + .csv) ============
for s in SPLIT_N:
    np.save(f'{OUT_DIR}/patches_{s}.npy', Xs[s])                    # float16, shape (N, 40, 40, 40)
    np.save(f'{OUT_DIR}/masks_{s}.npy', Ms[s])                      # uint8 0/1, same shape
    df_all[df_all.split == s].to_csv(f'{OUT_DIR}/index_{s}.csv', index=False)   # row i  <->  patches_{s}[i]
df_all.to_csv(f'{OUT_DIR}/index_all.csv', index=False)

config = dict(SEED=SEED, N_PARENTS=N_PARENTS, N_PER_CLASS=N_PER_CLASS, PARENT=PARENT, CELL=CELL, STORE=STORE,
              V_RANGE=V_RANGE, CONTRAST_RANGE=CONTRAST_RANGE, BIN_EDGES=BIN_EDGES, SNR_TIERS=SNR_TIERS,
              PSF_RANGE=PSF_RANGE, MATRIX_RANGE=MATRIX_RANGE, SUBTYPES={str(k): v for k, v in SUBTYPES.items()},
              CLASS_NAMES=CLASS_NAMES, SPLIT_N=SPLIT_N)
json.dump(config, open(f'{OUT_DIR}/data_config.json', 'w'), indent=2)
print('Saved to', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    if os.path.isfile(p):
        print(f'  {f:24s} {os.path.getsize(p)/1e6:8.1f} MB')

## Cell 7 - Validation and report figures
Proves the data meets the brief and saves 4 ready-to-use figures to `figures/` on Drive. **Screenshot the printed output - it is evidence for your report.**

In [ ]:
# ============ CELL 7: VALIDATION CHECKS + REPORT FIGURES ============
print('--- 1. Patches per split and class (brief: 5,250 / 1,125 / 1,125 ; 2,500 per class) ---')
print(pd.crosstab(df_all.split, df_all.class_name, margins=True))

print('\n--- 2. Volume-disjoint check (no parent volume may appear in two splits) ---')
ids = {s: set(df_all[df_all.split == s].parent_id) for s in SPLIT_N}
print({s: len(v) for s, v in ids.items()}, 'parents per split')
assert not (ids['train'] & ids['val']) and not (ids['train'] & ids['test']) and not (ids['val'] & ids['test'])
print('OK: zero parent volumes shared between train / val / test')

print('\n--- 3. Noise tiers per split (needed for the robustness experiment) ---')
print(pd.crosstab(df_all.split, df_all.snr_tier))

print('\n--- 4. Contrast bins of inclusions per split (needed for the contrast-stratified results) ---')
print(pd.crosstab(df_all[df_all.label == 2].split, df_all[df_all.label == 2].contrast_bin))

print('\n--- 5. Is defect size a shortcut? (KS test on defect volume; p > 0.05 = distributions indistinguishable) ---')
v = {k: df_all[df_all.label == i].actual_volume.values for i, k in enumerate(CLASS_NAMES)}
for a, b in [('pore', 'crack'), ('pore', 'inclusion'), ('crack', 'inclusion')]:
    st, p = ks_2samp(v[a], v[b]); print(f'  {a:6s} vs {b:9s}: KS = {st:.3f}, p = {p:.3f}')
print('  median volume (voxels):', {k: int(np.median(x)) for k, x in v.items()})

print('\n--- 6. Contamination check: voxels of OTHER defects inside a patch ---')
print('  total foreign voxels =', int(df_all.foreign_voxels.sum()), '(should be 0)')

# ---------- figures for the report ----------
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for i, k in enumerate(CLASS_NAMES):
    ax[0].hist(v[k], bins=40, alpha=0.5, label=k)
ax[0].set_xlabel('defect volume (voxels)'); ax[0].set_ylabel('count'); ax[0].legend()
ax[0].set_title('Same size distribution for all classes\n(fill fraction is NOT a shortcut)')
inc = df_all[df_all.label == 2]
ax[1].hist(inc.contrast, bins=40, color='C2'); ax[1].axvline(0, color='k', ls='--', lw=0.8)
ax[1].set_xlabel('inclusion contrast  (matrix = 0, void = -1)'); ax[1].set_title('Inclusion attenuation contrast')
cnt = pd.crosstab(df_all.split, df_all.snr_tier)[list(SNR_TIERS)]
cnt.plot(kind='bar', ax=ax[2], rot=0); ax[2].set_title('Patches per split and noise tier'); ax[2].set_ylabel('patches')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/data_overview.png', dpi=150); plt.show()

plot_gallery(Xs['train'], df_all[df_all.split == 'train'].reset_index(drop=True), per_row=5,
             fname=f'{OUT_DIR}/figures/gallery_classes.png', seed=3)
plot_3planes(Xs['train'], Ms['train'], df_all[df_all.split == 'train'].reset_index(drop=True),
             fname=f'{OUT_DIR}/figures/three_planes.png', seed=5)

# the same defect at three noise levels (parents differ, so we just show one example per tier)
fig, ax = plt.subplots(1, 3, figsize=(10, 3.4))
d_tr = df_all[df_all.split == 'train'].reset_index(drop=True)
for a, tier in zip(ax, SNR_TIERS):
    i = d_tr[(d_tr.label == 2) & (d_tr.snr_tier == tier) & (d_tr.contrast > 0.4)].index[0]
    a.imshow(view(Xs['train'][i], d_tr.bg_level[i])[STORE // 2], cmap='gray', vmin=0, vmax=1.8)
    a.set_title(f'SNR tier: {tier} (SNR={SNR_TIERS[tier]})', fontsize=9); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/noise_tiers.png', dpi=150); plt.show()

# ---------- re-load from disk to prove the files are readable ----------
chk = np.load(f'{OUT_DIR}/patches_test.npy', mmap_mode='r'); idx = pd.read_csv(f'{OUT_DIR}/index_test.csv')
print('\nReload check -> patches_test.npy', chk.shape, chk.dtype, '| index_test.csv', idx.shape)
assert chk.shape[0] == len(idx) == SPLIT_N['test']
print('ALL CHECKS PASSED - data generation complete.')

## Done. What you should have on Drive
`defect_project/data/` -> `patches_{train,val,test}.npy`, `masks_{train,val,test}.npy`, `index_{train,val,test}.csv`, `index_all.csv`, `data_config.json`, `figures/` (4 PNGs).

**Next: Block 2** - preprocessing and the 3D CNN.